# **Teachable Machine**

En esta práctica se debe exportar el modelo que se genera en Teachable Machine.

A la hora de exportar el modelo, se puede exportar para Tensorflow.

También viene un fragmento de código para usar el modelo:

```
from keras.models import load_model  # TensorFlow is required for Keras to work
from PIL import Image, ImageOps  # Install pillow instead of PIL
import numpy as np

# Disable scientific notation for clarity
np.set_printoptions(suppress=True)

# Load the model
model = load_model("keras_Model.h5", compile=False)

# Load the labels
class_names = open("labels.txt", "r").readlines()

# Create the array of the right shape to feed into the keras model
# The 'length' or number of images you can put into the array is
# determined by the first position in the shape tuple, in this case 1
data = np.ndarray(shape=(1, 224, 224, 3), dtype=np.float32)

# Replace this with the path to your image
image = Image.open("<IMAGE_PATH>").convert("RGB")

# resizing the image to be at least 224x224 and then cropping from the center
size = (224, 224)
image = ImageOps.fit(image, size, Image.Resampling.LANCZOS)

# turn the image into a numpy array
image_array = np.asarray(image)

# Normalize the image
normalized_image_array = (image_array.astype(np.float32) / 127.5) - 1

# Load the image into the array
data[0] = normalized_image_array

# Predicts the model
prediction = model.predict(data)
index = np.argmax(prediction)
class_name = class_names[index]
confidence_score = prediction[0][index]

# Print prediction and confidence score
print("Class:", class_name[2:], end="")
print("Confidence Score:", confidence_score)

```

**Tarea**: subir 3 imágenes propias (una clara, una ambigua, una que no pertenezca a ninguna categoría) y anotar qué predice y con qué confianza.

**¡Ojo! Hay que cargar una  versión anterior de Keras**

In [1]:
# forzar TensorFlow a usar el comportamiento clásico de Keras 2
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import tensorflow as tf
import tf_keras as keras

print("Versión de TensorFlow:", tf.__version__)
print("Versión de TF-Keras (Legacy):", keras.__version__)

try:
    # Cargamos el modelo utilizando el módulo de compatibilidad oficial tf_keras
    model = keras.models.load_model('keras_model.h5', compile=False)
    print("\n¡MODELO CARGADO CON ÉXITO!")
    model.summary()
except Exception as e:
    print(f"\nError al cargar: {e}")

Versión de TensorFlow: 2.21.0
Versión de TF-Keras (Legacy): 2.21.0

¡MODELO CARGADO CON ÉXITO!
Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sequential_1 (Sequential)   (None, 1280)              410208    
                                                                 
 sequential_3 (Sequential)   (None, 2)                 128300    
                                                                 
Total params: 538508 (2.05 MB)
Trainable params: 524428 (2.00 MB)
Non-trainable params: 14080 (55.00 KB)
_________________________________________________________________


In [5]:
import tensorflow as tf
from keras.models import load_model
from PIL import Image, ImageOps
import numpy as np

class_names = open("labels.txt", "r").readlines()
data = np.ndarray(shape=(1, 224, 224, 3), dtype=np.float32)
image = Image.open("WIN_20261103_09_13_14_Pro.jpg").convert("RGB")
size = (224, 224)
image = ImageOps.fit(image, size, Image.Resampling.LANCZOS)
image_array = np.asarray(image)
normalized_image_array = (image_array.astype(np.float32) / 127.5) - 1
data[0] = normalized_image_array

prediction = model.predict(data)
index = np.argmax(prediction)
class_name = class_names[index]
confidence_score = prediction[0][index]

# Print prediction and confidence score
print("Class:", class_name[2:], end="")
print("Confidence Score:", confidence_score)

1/1 [==============================] - 1s 977ms/step
Class: Class 1
Confidence Score: 0.99973613


In [6]:
def clasificar(nombre_foto):
    datos = np.ndarray(shape=(1, 224, 224, 3), dtype=np.float32)
    foto = Image.open(nombre_foto).convert("RGB")
    foto = ImageOps.fit(foto, (224, 224), Image.Resampling.LANCZOS)
    foto_array = np.asarray(foto)
    datos[0] = (foto_array.astype(np.float32) / 127.5) - 1

    prediccion = model.predict(datos)
    indice = np.argmax(prediccion)
    print("Foto:", nombre_foto)
    print("Clase:", class_names[indice][2:])
    print("Confianza:", prediccion[0][indice])

clasificar("WIN_20261103_09_13_14_Pro.jpg")

1/1 [==============================] - 0s 40ms/step
Foto: WIN_20261103_09_13_14_Pro.jpg
Clase: Class 1

Confianza: 0.99973613


In [7]:
clasificar("WIN_20261103_09_13_33_Pro.jpg")

1/1 [==============================] - 0s 36ms/step
Foto: WIN_20261103_09_13_33_Pro.jpg
Clase: Class 2

Confianza: 0.98388666


In [8]:
clasificar("WIN_20261103_09_13_54_Pro.jpg")

1/1 [==============================] - 0s 38ms/step
Foto: WIN_20261103_09_13_54_Pro.jpg
Clase: Class 1

Confianza: 0.9865454


**¿Cómo se usa el modelo exportado? ¿Cómo se comporta?**

---



Cargamos el modelo (load_model) y sus clases (labels.txt). Cada foto se ajusta a 224x224, se normaliza y se pasa por model.predict para seleccionar la clase con mayor porcentaje.

Los resultados muestran que acierta con la foto clara (99,97 % en Class 1), pero le da un 98,39 % (Class 2) a la ambigua y un 98,65 % (Class 1) a una foto que no tiene nada que ver.
La red solo conoce dos clases y está forzada a repartir el 100 % de probabilidad entre ellas. Al no tener la opción "ninguna", responderá siempre con un porcentaje muy alto aunque no tenga ni idea de lo que está viendo.